In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
from sklearn.impute import SimpleImputer
from tqdm import tqdm
import matplotlib.pyplot as plt

In [14]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# ------------------------------
# CONFIG
# ------------------------------
FEATURES_DIR = "features"

# Keep only ECG, GSR (foot), and Resp features
KEEP_FEATURES = [
    "HR_mean","SDNN","RMSSD","pNN50","LF_HF",
    "RR_mean","RR_std","RR_min","RR_max",
    "GSR_foot_mean","GSR_foot_std","GSR_foot_peaks","GSR_foot_slope",
    "Resp_rate","Resp_std","Resp_power","Resp_peak_count"
]

# ------------------------------
# Load Data
# ------------------------------
dfs = []
for file in os.listdir(FEATURES_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(FEATURES_DIR, file))
        dfs.append(df)

# data = pd.concat(dfs, ignore_index=True)

data = pd.read_csv("stress_features_all.csv")

# ------------------------------
# Keep only useful features + label
# ------------------------------
# Some recordings might miss entire columns -> reindex to ensure consistency
all_cols = KEEP_FEATURES + ["label"]
data = data.reindex(columns=all_cols)

# Drop columns that are fully NaN across all recordings
data = data.dropna(axis=1, how="all")

# Separate X and y
X = data.drop(columns=["label"])
y = data["label"]

# ------------------------------
# Handle Missing Values
# ------------------------------
# Fill missing values in numeric columns only
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].mean())

# ------------------------------
# Encode labels
# ------------------------------
le = LabelEncoder()
y = le.fit_transform(y)   # "low", "medium", "high" -> 0,1,2

# ------------------------------
# Train/Test Split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ------------------------------
# Train Random Forest
# ------------------------------
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
clf.fit(X_train, y_train)

# ------------------------------
# Evaluation
# ------------------------------
y_pred = clf.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Cross-validation (5-fold)
cv_scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
print(f"CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# ------------------------------
# Feature Importances
# ------------------------------
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop Feature Importances:")
print(importances.head(10))


c:\Users\prana\anaconda3\envs\tds\lib\site-packages\pandas\core\generic.py:2153: RuntimeWarning: overflow encountered in cast
  arr = np.asarray(values, dtype=dtype)


ValueError: Input X contains infinity or a value too large for dtype('float32').